In [7]:
from ollama import Client
from ollama import ChatResponse
from typing import Optional, Any


def generate_ollama_ans(
    model: str,
    user_query: str,
    context: Optional[str] = None,
    system_prompt: Optional[str] = None,
    num_ctx: int = 8192,
    host: str = "http://localhost:11434",
    think: Optional[bool | str] = None,
    format: Optional[str | dict] = None,
    **kwargs: Any
) -> ChatResponse:
    """
    Generate a response using a local Ollama model.

    Args:
        model: Name of the Ollama model.
        user_query: Question to answer.
        context: Optional ontology or other reasoning context.
        system_prompt: System instructions.
        num_ctx: Context window size.
        host: Ollama server address.
        think: Enable/disable reasoning, or specify a supported level.
        format: Optional JSON mode or JSON schema.
        **kwargs: Additional Ollama inference options.

    Returns:
        Complete Ollama ChatResponse.
    """

    client = Client(host=host)

    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    user_content = user_query

    if context is not None:
        user_content = f"""
<ontology>
{context}
</ontology>

<question>
{user_query}
</question>
"""

    messages.append({
        "role": "user",
        "content": user_content
    })

    options = {
        **kwargs,
        "num_ctx": num_ctx
    }

    response = client.chat(
        model=model,
        messages=messages,
        options=options,
        think=think,
        format=format,
        stream=False
    )

    return response

In [12]:
MODEL = "gemma4:e4b"

with open("../univ-bench-inst_no_individuals.txt", encoding="utf-8") as f:
    lubm_onto_full = f.read()

system_prompt = """
You are an expert in OWL ontology reasoning.

Answer questions using only the provided ontology.
Apply OWL semantics and the Open World Assumption.
Do not rely on external knowledge.

Return a JSON object containing:
- answer: yes, no, or unresolvable
- justification:formal justification for the inference
"""


from pydantic import BaseModel, Field
from typing import Literal

class OntologyAnswer(BaseModel):
    answer: Literal["yes", "no"]
    justification: str = Field(description="justification for the inference")
   

response = generate_ollama_ans(
    model=MODEL,
    user_query="Is every UndergraduateStudent a Student?",
    context=lubm_onto_full,
    system_prompt=system_prompt,
    num_ctx=32768,
    think=True,
    format="json",
    temperature=0.6,
    top_p=0.95,
    num_predict=-1,
    seed=42
)
# Risposta strutturata
result = OntologyAnswer.model_validate_json(
    response.message.content
)

# Thinking, quando disponibile
thinking = response.message.thinking


In [10]:
result

OntologyAnswer(answer='yes', justification='The class definition for UndergraduateStudent explicitly includes the statement :UndergraduateStudent rdf:type owl:Class ; rdfs:subClassOf :Student ; rdfs:label "undergraduate student". This establishes a subclass relationship where every member of UndergraduateStudent is necessarily a Student.')

In [11]:
thinking

'The user is asking if the class `UndergraduateStudent` is a subclass of the class `Student` based on the provided OWL ontology.\n\n1.  **Locate `UndergraduateStudent`:**\n    `:UndergraduateStudent rdf:type owl:Class ; rdfs:subClassOf :Student ; rdfs:label "undergraduate student"`\n2.  **Locate `Student`:**\n    `:Student rdf:type owl:Class ; owl:equivalentClass [...] ; rdfs:label "student"`\n3.  **Check the relationship:** The definition for `UndergraduateStudent` explicitly states `rdfs:subClassOf :Student`.\n\nTherefore, based on the ontology definition, every instance of `UndergraduateStudent` must be an instance of `Student`.'